In [0]:
CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

def gold(name, sql):
    spark.sql(f"CREATE OR REPLACE TABLE gold.{name} AS {sql}")
    print(name, "->", spark.table(f"gold.{name}").count())

In [0]:
gold("fraud_by_day_of_week", """
SELECT day_of_week, day_of_week_num,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount
FROM silver.transactions WHERE is_labeled
GROUP BY day_of_week, day_of_week_num
""")

gold("fraud_by_time_of_day", """
SELECT time_of_day, transaction_hour,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount
FROM silver.transactions WHERE is_labeled
GROUP BY time_of_day, transaction_hour
""")

fraud_by_day_of_week -> 7
fraud_by_time_of_day -> 24


In [0]:
gold("fraud_daily", """
SELECT transaction_date, year_month,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_loss,
       COUNT(DISTINCT CASE WHEN is_fraud THEN client_id END) AS unique_fraud_users
FROM silver.transactions WHERE is_labeled
GROUP BY transaction_date, year_month
""")

gold("fraud_monthly", """
SELECT year_month,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_loss
FROM silver.transactions WHERE is_labeled
GROUP BY year_month
""")

fraud_daily -> 3591
fraud_monthly -> 118


In [0]:
gold("fraud_by_mcc", """
SELECT t.mcc, COALESCE(t.mcc_description, 'Unknown') AS mcc_description,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN t.is_fraud THEN t.amount ELSE 0 END) AS total_fraud_amount,
       AVG(CASE WHEN t.is_fraud THEN t.amount END) AS avg_fraud_amount
FROM silver.transactions t WHERE t.is_labeled
GROUP BY t.mcc, t.mcc_description
HAVING COUNT(*) >= 100
""")

gold("fraud_by_merchant", """
SELECT merchant_id, merchant_city, merchant_state,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS total_fraud_amount
FROM silver.transactions WHERE is_labeled
GROUP BY merchant_id, merchant_city, merchant_state
HAVING SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) > 0
""")

fraud_by_mcc -> 109
fraud_by_merchant -> 1226


In [0]:
gold("fraud_by_user", """
SELECT t.client_id,
       u.current_age, u.gender, u.credit_score, u.yearly_income,
       COUNT(*) AS total_txn,
       SUM(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
       ROUND(100.0 * SUM(CASE WHEN t.is_fraud THEN 1 ELSE 0 END) / COUNT(*), 4) AS fraud_rate_pct,
       SUM(CASE WHEN t.is_fraud THEN t.amount ELSE 0 END) AS total_fraud_amount,
       MIN(CASE WHEN t.is_fraud THEN t.transaction_ts END) AS first_fraud_ts
FROM silver.transactions t
LEFT JOIN silver.users u ON t.client_id = u.client_id
WHERE t.is_labeled
GROUP BY t.client_id, u.current_age, u.gender, u.credit_score, u.yearly_income
""")

gold("user_weekly_spend", """
WITH w AS (
  SELECT client_id, week_start,
         COUNT(*) AS txn_count,
         SUM(amount) AS week_amount,
         SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_txn,
         COUNT(DISTINCT CASE WHEN is_fraud THEN client_id END) AS is_fraud_user
  FROM silver.transactions WHERE is_labeled
  GROUP BY client_id, week_start
)
SELECT *,
       AVG(week_amount) OVER (PARTITION BY client_id) AS user_avg_week_amount,
       ROUND(week_amount / NULLIF(AVG(week_amount) OVER (PARTITION BY client_id), 0), 3) AS spend_ratio_vs_avg
FROM w
""")

fraud_by_user -> 1219
user_weekly_spend -> 606418


In [0]:
gold("amount_profile", """
SELECT is_fraud,
       CASE WHEN amount < 50 THEN '1. <$50'
            WHEN amount < 200 THEN '2. $50-200'
            WHEN amount < 500 THEN '3. $200-500'
            WHEN amount < 1000 THEN '4. $500-1000'
            ELSE '5. $1000+' END AS amount_band,
       COUNT(*) AS txn_count,
       AVG(amount) AS avg_amount,
       PERCENTILE_APPROX(amount, 0.5) AS median_amount,
       SUM(amount) AS total_amount
FROM silver.transactions
WHERE is_labeled AND amount > 0
GROUP BY is_fraud, amount_band
""")

amount_profile -> 10


In [0]:
%sql
SELECT ROUND(100.0*SUM(fraud_txn)/SUM(total_txn),4) AS overall_fraud_rate_pct,
       SUM(total_txn) AS labeled_txn
FROM gold.fraud_daily

overall_fraud_rate_pct,labeled_txn
0.1495,8914963
